In [ ]:
# @title Dados dos problemas e imports

import heapq
import time
import json
import random
import numpy as np


# ============================================================
# 8-PUZZLE - DADOS
# ============================================================

# Tamanho do tabuleiro
N_PUZZLE = 3


def criar_estado_objetivo():
    estado = list(range(1, N_PUZZLE * N_PUZZLE))
    estado.append(0)
    return tuple(estado)


def estado_soluvel(estado):
    pecas = [peca for peca in estado if peca != 0]
    inversoes = 0

    for i in range(len(pecas)):
        for j in range(i + 1, len(pecas)):
            if pecas[i] > pecas[j]:
                inversoes += 1

    if N_PUZZLE % 2 == 1:
        return inversoes % 2 == 0

    posicao_zero = estado.index(0)
    linha = posicao_zero // N_PUZZLE
    linha_do_vazio = N_PUZZLE - linha

    if linha_do_vazio % 2 == 0:
        return inversoes % 2 == 1

    return inversoes % 2 == 0


def gerar_estado_inicial():
    pecas = list(range(N_PUZZLE * N_PUZZLE))

    while True:

        estado = []
        pecas_disponiveis = pecas.copy()

        for _ in range(N_PUZZLE * N_PUZZLE):

            peca = random.choice(pecas_disponiveis)

            estado.append(peca)
            pecas_disponiveis.remove(peca)

        estado = tuple(estado)

        if estado_soluvel(estado):
            return estado


ESTADO_OBJETIVO = criar_estado_objetivo()
ESTADO_INICIAL = gerar_estado_inicial()


# ============================================================
# TSP - DADOS
# ============================================================

# Arquivo JSON gerado pelo conversor dos arquivos TSPLIB
ARQUIVO_TSP = "ulysses16.json"

with open(ARQUIVO_TSP, "r") as arquivo:
    tsp = json.load(arquivo)


random_seed = None

rng = np.random.default_rng(random_seed)


distancias = tsp["distance_matrix"]
coordenadas = tsp["coordinates"]
cidades = tsp["cities"]

N_TSP = len(cidades)

# Cidade inicial do TSP
inicio = 0


print("Dados carregados.")
print(f"8-Puzzle: {N_PUZZLE}x{N_PUZZLE}")
print(f"TSP: {tsp['name']}")
print(f"Número de cidades: {N_TSP}")
print(f"Cidade inicial do TSP: {inicio}")

Dados carregados.
8-Puzzle: 3x3
TSP: ulysses16.tsp
Número de cidades: 16
Cidade inicial do TSP: 0


In [ ]:
# @title Definição das buscas


# ============================================================
# ============================================================
# 8-PUZZLE - BUSCAS HEURÍSTICAS
# ============================================================
# ============================================================


# ------------------------------------------------------------
# Heurística - Distância de Manhattan
# ------------------------------------------------------------

def heuristica_puzzle(estado):

    distancia = 0

    for posicao in range(N_PUZZLE * N_PUZZLE):

        peca = estado[posicao]

        if peca == 0:
            continue

        posicao_objetivo = peca - 1

        linha = posicao // N_PUZZLE
        coluna = posicao % N_PUZZLE

        linha_objetivo = (
            posicao_objetivo // N_PUZZLE
        )

        coluna_objetivo = (
            posicao_objetivo % N_PUZZLE
        )

        distancia += abs(
            linha - linha_objetivo
        )

        distancia += abs(
            coluna - coluna_objetivo
        )

    return distancia


# ------------------------------------------------------------
# Gerar sucessores
# ------------------------------------------------------------

def sucessores_puzzle(estado):

    sucessores = []

    posicao_zero = estado.index(0)

    linha = posicao_zero // N_PUZZLE
    coluna = posicao_zero % N_PUZZLE

    movimentos = []

    if linha > 0:
        movimentos.append(
            posicao_zero - N_PUZZLE
        )

    if linha < N_PUZZLE - 1:
        movimentos.append(
            posicao_zero + N_PUZZLE
        )

    if coluna > 0:
        movimentos.append(
            posicao_zero - 1
        )

    if coluna < N_PUZZLE - 1:
        movimentos.append(
            posicao_zero + 1
        )

    for nova_posicao in movimentos:

        novo_estado = list(estado)

        novo_estado[posicao_zero], novo_estado[nova_posicao] = (
            novo_estado[nova_posicao],
            novo_estado[posicao_zero]
        )

        sucessores.append(
            tuple(novo_estado)
        )

    return sucessores


# ------------------------------------------------------------
# Busca Gulosa e A*
# ------------------------------------------------------------

def buscar_puzzle(busca="astar"):

    borda = []

    heapq.heappush(
        borda,
        (
            0,
            0,
            ESTADO_INICIAL,
            [ESTADO_INICIAL]
        )
    )

    visitados = set()
    expandidos = 0

    while borda:

        _, custo, estado, caminho = (
            heapq.heappop(borda)
        )

        if estado in visitados:
            continue

        visitados.add(estado)
        expandidos += 1

        if estado == ESTADO_OBJETIVO:

            return {
                "caminho": caminho,
                "custo": custo,
                "expandidos": expandidos
            }

        for proximo in sucessores_puzzle(estado):

            if proximo in visitados:
                continue

            novo_custo = custo + 1
            h = heuristica_puzzle(proximo)

            if busca == "gulosa":
                f = h
            else:
                f = novo_custo + h

            novo_caminho = (
                caminho + [proximo]
            )

            heapq.heappush(
                borda,
                (
                    f,
                    novo_custo,
                    proximo,
                    novo_caminho
                )
            )

    return None


# ------------------------------------------------------------
# IDA*
# ------------------------------------------------------------

def ida_busca(
    estado,
    custo,
    limite,
    caminho,
    visitados,
    contador
):

    contador["expandidos"] += 1

    f = (
        custo +
        heuristica_puzzle(estado)
    )

    if f > limite:
        return f, None

    if estado == ESTADO_OBJETIVO:
        return custo, caminho

    menor_limite = float("inf")

    for proximo in sucessores_puzzle(estado):

        if proximo in visitados:
            continue

        visitados.add(proximo)
        caminho.append(proximo)

        resultado, solucao = ida_busca(
            proximo,
            custo + 1,
            limite,
            caminho,
            visitados,
            contador
        )

        if solucao is not None:
            return resultado, solucao

        if resultado < menor_limite:
            menor_limite = resultado

        caminho.pop()
        visitados.remove(proximo)

    return menor_limite, None


def buscar_ida_star():

    limite = heuristica_puzzle(
        ESTADO_INICIAL
    )

    contador = {
        "expandidos": 0
    }

    while True:

        visitados = {
            ESTADO_INICIAL
        }

        caminho = [
            ESTADO_INICIAL
        ]

        resultado, solucao = ida_busca(
            ESTADO_INICIAL,
            0,
            limite,
            caminho,
            visitados,
            contador
        )

        if solucao is not None:

            return {
                "caminho": solucao,
                "custo": len(solucao) - 1,
                "expandidos": contador["expandidos"]
            }

        if resultado == float("inf"):
            return None

        limite = resultado


# ============================================================
# ============================================================
# TSP - BUSCAS LOCAIS
# ============================================================
# ============================================================


# ------------------------------------------------------------
# Custo da rota
# ------------------------------------------------------------

def custo_rota(rota):

    custo = 0

    for i in range(len(rota) - 1):

        custo += distancias[
            rota[i]
        ][
            rota[i + 1]
        ]

    return custo


# ------------------------------------------------------------
# Gerar vizinho
# ------------------------------------------------------------

def gerar_vizinho(rota, rng):

    vizinho = rota.copy()

    i, j = rng.choice(
        np.arange(
            1,
            len(vizinho) - 1
        ),
        size=2,
        replace=False
    )

    vizinho[i], vizinho[j] = (
        vizinho[j],
        vizinho[i]
    )

    return vizinho


# ------------------------------------------------------------
# Hill Climbing
# ------------------------------------------------------------

def subida_de_encosta(
    rota_inicial,
    iter_max=600
):

    rota_atual = rota_inicial.copy()

    custo_atual = custo_rota(
        rota_atual
    )

    for _ in range(iter_max):

        vizinhos = []

        for i in range(
            1,
            len(rota_atual) - 1
        ):

            for j in range(
                i + 1,
                len(rota_atual) - 1
            ):

                vizinho = rota_atual.copy()

                vizinho[i], vizinho[j] = (
                    vizinho[j],
                    vizinho[i]
                )

                vizinhos.append(
                    vizinho
                )

        melhor_vizinho = min(
            vizinhos,
            key=custo_rota
        )

        custo_vizinho = custo_rota(
            melhor_vizinho
        )

        if custo_vizinho >= custo_atual:

            return (
                rota_atual,
                custo_atual
            )

        rota_atual = melhor_vizinho
        custo_atual = custo_vizinho

    return (
        rota_atual,
        custo_atual
    )


# ------------------------------------------------------------
# Simulated Annealing
# ------------------------------------------------------------

def simulated_annealing(
    rota_inicial,
    temperatura_inicial=1000,
    taxa_resfriamento=0.995,
    temperatura_minima=1e-3,
    iter_max=600
):

    rota_atual = rota_inicial.copy()

    custo_atual = custo_rota(
        rota_atual
    )

    melhor_rota = rota_atual.copy()
    melhor_custo = custo_atual

    temperatura = temperatura_inicial

    for _ in range(iter_max):

        if temperatura <= temperatura_minima:
            break

        vizinho = gerar_vizinho(
            rota_atual,
            rng
        )

        custo_vizinho = custo_rota(
            vizinho
        )

        delta = (
            custo_vizinho -
            custo_atual
        )

        if (
            delta < 0
            or
            rng.random() <
            np.exp(
                -delta /
                temperatura
            )
        ):

            rota_atual = vizinho
            custo_atual = custo_vizinho

            if custo_atual < melhor_custo:

                melhor_rota = (
                    rota_atual.copy()
                )

                melhor_custo = (
                    custo_atual
                )

        temperatura *= (
            taxa_resfriamento
        )

    return (
        melhor_rota,
        melhor_custo
    )


# ------------------------------------------------------------
# Simulated Annealing com Reaquecimento
# ------------------------------------------------------------

def simulated_annealing_reaquecimento(
    rota_inicial,
    temperatura_inicial=1000,
    taxa_resfriamento=0.995,
    temperatura_minima=1e-3,
    iter_max=600,
    reaquecimento=3
):

    rota_atual = rota_inicial.copy()

    custo_atual = custo_rota(
        rota_atual
    )

    melhor_rota = rota_atual.copy()
    melhor_custo = custo_atual

    temperatura = temperatura_inicial

    # Conta quantas vezes a temperatura
    # atingiu o limite mínimo.
    reaquecimentos_realizados = 0

    for _ in range(iter_max):

        # ----------------------------------------------------
        # Reaquecimento
        # ----------------------------------------------------

        if temperatura <= temperatura_minima:

            if (
                reaquecimentos_realizados
                >= reaquecimento
            ):
                break

            temperatura = temperatura_inicial

            reaquecimentos_realizados += 1

        # ----------------------------------------------------
        # Gerar vizinho
        # ----------------------------------------------------

        vizinho = gerar_vizinho(
            rota_atual,
            rng
        )

        custo_vizinho = custo_rota(
            vizinho
        )

        delta = (
            custo_vizinho -
            custo_atual
        )

        # ----------------------------------------------------
        # Critério de aceitação
        # ----------------------------------------------------

        if (
            delta < 0
            or
            rng.random() <
            np.exp(
                -delta /
                temperatura
            )
        ):

            rota_atual = vizinho
            custo_atual = custo_vizinho

            # Atualiza a melhor solução
            if custo_atual < melhor_custo:

                melhor_rota = (
                    rota_atual.copy()
                )

                melhor_custo = (
                    custo_atual
                )

        # ----------------------------------------------------
        # Resfriamento
        # ----------------------------------------------------

        temperatura *= (
            taxa_resfriamento
        )

    return (
        melhor_rota,
        melhor_custo,
        reaquecimentos_realizados
    )

In [ ]:
# @title Executar buscas heurísticas - 8-Puzzle


def imprimir_estado_puzzle(estado):

    for linha in range(N_PUZZLE):

        inicio_linha = (
            linha * N_PUZZLE
        )

        fim_linha = (
            inicio_linha + N_PUZZLE
        )

        print(
            " ".join(
                "_" if valor == 0
                else str(valor)
                for valor in estado[
                    inicio_linha:fim_linha
                ]
            )
        )


print("\n" + "=" * 60)
print("BUSCAS HEURÍSTICAS - 8-PUZZLE")
print("=" * 60)

print("\nEstado inicial:")
imprimir_estado_puzzle(
    ESTADO_INICIAL
)

print("\nEstado objetivo:")
imprimir_estado_puzzle(
    ESTADO_OBJETIVO
)

print(
    "\nHeurística: "
    "Distância de Manhattan"
)


# ============================================================
# GULOSA
# ============================================================

t0 = time.perf_counter()

resultado_gulosa = buscar_puzzle(
    busca="gulosa"
)

tempo_gulosa = (
    time.perf_counter() - t0
) * 1000.0


print("\n[GULOSA]")

if resultado_gulosa is None:

    print(
        "Nenhuma solução encontrada."
    )

else:

    print(
        f"Custo: "
        f"{resultado_gulosa['custo']}"
    )

    print(
        f"Nós expandidos: "
        f"{resultado_gulosa['expandidos']}"
    )

    print(
        f"Tempo: "
        f"{tempo_gulosa:.3f} ms"
    )


# ============================================================
# A*
# ============================================================

t0 = time.perf_counter()

resultado_astar = buscar_puzzle(
    busca="astar"
)

tempo_astar = (
    time.perf_counter() - t0
) * 1000.0


print("\n[A*]")

if resultado_astar is None:

    print(
        "Nenhuma solução encontrada."
    )

else:

    print(
        f"Custo: "
        f"{resultado_astar['custo']}"
    )

    print(
        f"Nós expandidos: "
        f"{resultado_astar['expandidos']}"
    )

    print(
        f"Tempo: "
        f"{tempo_astar:.3f} ms"
    )


# ============================================================
# IDA*
# ============================================================

t0 = time.perf_counter()

resultado_ida = buscar_ida_star()

tempo_ida = (
    time.perf_counter() - t0
) * 1000.0


print("\n[IDA*]")

if resultado_ida is None:

    print(
        "Nenhuma solução encontrada."
    )

else:

    print(
        f"Custo: "
        f"{resultado_ida['custo']}"
    )

    print(
        f"Nós expandidos: "
        f"{resultado_ida['expandidos']}"
    )

    print(
        f"Tempo: "
        f"{tempo_ida:.3f} ms"
    )


# ============================================================
# MOSTRAR SOLUÇÕES
# ============================================================

def imprimir_solucao(
    nome,
    resultado
):

    if resultado is None:
        return

    print(
        f"\n{'-' * 40}"
    )

    print(
        f"Solução - {nome}"
    )

    print(
        f"{'-' * 40}"
    )

    for i, estado in enumerate(
        resultado["caminho"]
    ):

        print(
            f"\nPasso {i}:"
        )

        imprimir_estado_puzzle(
            estado
        )


imprimir_solucao(
    "Gulosa",
    resultado_gulosa
)

imprimir_solucao(
    "A*",
    resultado_astar
)

imprimir_solucao(
    "IDA*",
    resultado_ida
)


BUSCAS HEURÍSTICAS - 8-PUZZLE

Estado inicial:
8 7 2
4 1 3
5 _ 6

Estado objetivo:
1 2 3
4 5 6
7 8 _

Heurística: Distância de Manhattan

[GULOSA]
Custo: 31
Nós expandidos: 142
Tempo: 1.504 ms

[A*]
Custo: 19
Nós expandidos: 389
Tempo: 13.195 ms

[IDA*]
Custo: 19
Nós expandidos: 532
Tempo: 7.491 ms

----------------------------------------
Solução - Gulosa
----------------------------------------

Passo 0:
8 7 2
4 1 3
5 _ 6

Passo 1:
8 7 2
4 1 3
_ 5 6

Passo 2:
8 7 2
_ 1 3
4 5 6

Passo 3:
_ 7 2
8 1 3
4 5 6

Passo 4:
7 _ 2
8 1 3
4 5 6

Passo 5:
7 1 2
8 _ 3
4 5 6

Passo 6:
7 1 2
_ 8 3
4 5 6

Passo 7:
_ 1 2
7 8 3
4 5 6

Passo 8:
1 _ 2
7 8 3
4 5 6

Passo 9:
1 2 _
7 8 3
4 5 6

Passo 10:
1 2 3
7 8 _
4 5 6

Passo 11:
1 2 3
7 8 6
4 5 _

Passo 12:
1 2 3
7 8 6
4 _ 5

Passo 13:
1 2 3
7 _ 6
4 8 5

Passo 14:
1 2 3
_ 7 6
4 8 5

Passo 15:
1 2 3
4 7 6
_ 8 5

Passo 16:
1 2 3
4 7 6
8 _ 5

Passo 17:
1 2 3
4 7 6
8 5 _

Passo 18:
1 2 3
4 7 _
8 5 6

Passo 19:
1 2 3
4 _ 7
8 5 6

Passo 20:
1 2 3
4 5 7
8 _ 6


In [ ]:
# @title Executar buscas locais - TSP


# ============================================================
# GERAR ROTA INICIAL
# ============================================================

rota_inicial = list(cidades)

rota_inicial.remove(
    inicio
)

rng.shuffle(
    rota_inicial
)

rota_inicial = (
    [inicio]
    +
    rota_inicial
    +
    [inicio]
)


# ============================================================
# INFORMAÇÕES DO PROBLEMA
# ============================================================

print("=" * 60)
print("BUSCAS LOCAIS - TSP")
print("=" * 60)

print(
    f"\nProblema: "
    f"{tsp['name']}"
)

print(
    f"Número de cidades: "
    f"{N_TSP}"
)

print(
    "\nRota inicial:"
)

print(
    " -> ".join(
        map(str, rota_inicial)
    )
)

print(
    f"Custo inicial: "
    f"{custo_rota(rota_inicial)}"
)


# ============================================================
# HILL CLIMBING
# ============================================================

tempo_inicial = (
    time.perf_counter()
)

rota_hc, custo_hc = (
    subida_de_encosta(
        rota_inicial
    )
)

tempo_hc = (
    time.perf_counter() -
    tempo_inicial
) * 1000.0


print(
    "\n[HILL CLIMBING]"
)

print(
    "Rota:",
    " -> ".join(
        map(str, rota_hc)
    )
)

print(
    f"Custo: {custo_hc}"
)

print(
    f"Tempo: "
    f"{tempo_hc:.3f} ms"
)


# ============================================================
# SIMULATED ANNEALING
# ============================================================

tempo_inicial = (
    time.perf_counter()
)

rota_sa, custo_sa = (
    simulated_annealing(
        rota_inicial
    )
)

tempo_sa = (
    time.perf_counter() -
    tempo_inicial
) * 1000.0


print(
    "\n[SIMULATED ANNEALING]"
)

print(
    "Rota:",
    " -> ".join(
        map(str, rota_sa)
    )
)

print(
    f"Custo: {custo_sa}"
)

print(
    f"Tempo: "
    f"{tempo_sa:.3f} ms"
)


# ============================================================
# SIMULATED ANNEALING COM REAQUECIMENTO
# ============================================================

tempo_inicial = (
    time.perf_counter()
)

(
    rota_sa_r,
    custo_sa_r,
    reaquecimentos
) = simulated_annealing_reaquecimento(
    rota_inicial
)

tempo_sa_r = (
    time.perf_counter() -
    tempo_inicial
) * 1000.0


print(
    "\n[SIMULATED ANNEALING "
    "COM REAQUECIMENTO]"
)

print(
    "Rota:",
    " -> ".join(
        map(str, rota_sa_r)
    )
)

print(
    f"Custo: {custo_sa_r}"
)

print(
    f"Tempo: "
    f"{tempo_sa_r:.3f} ms"
)

print(
    f"Reaquecimentos realizados: "
    f"{reaquecimentos}"
)

BUSCAS LOCAIS - TSP

Problema: ulysses16.tsp
Número de cidades: 16

Rota inicial:
0 -> 10 -> 5 -> 1 -> 12 -> 6 -> 9 -> 15 -> 3 -> 4 -> 2 -> 14 -> 8 -> 7 -> 11 -> 13 -> 0
Custo inicial: 14221

[HILL CLIMBING]
Rota: 0 -> 15 -> 12 -> 13 -> 5 -> 6 -> 11 -> 7 -> 3 -> 1 -> 2 -> 9 -> 8 -> 10 -> 4 -> 14 -> 0
Custo: 7697
Tempo: 5.852 ms

[SIMULATED ANNEALING]
Rota: 0 -> 11 -> 12 -> 13 -> 14 -> 4 -> 10 -> 8 -> 9 -> 5 -> 6 -> 15 -> 2 -> 1 -> 3 -> 7 -> 0
Custo: 6860
Tempo: 25.068 ms

[SIMULATED ANNEALING COM REAQUECIMENTO]
Rota: 0 -> 15 -> 12 -> 8 -> 10 -> 4 -> 14 -> 6 -> 5 -> 9 -> 11 -> 13 -> 3 -> 1 -> 2 -> 7 -> 0
Custo: 7577
Tempo: 27.024 ms
Reaquecimentos realizados: 0
